# DỰ ĐOÁN BỆNH TIỂU ĐƯỜNG — 3 MÔ HÌNH ML CƠ BẢN

**Assignment 03 — Neural Networks and Representation Learning**

**Bài toán:** Phân loại nhị phân — so sánh 3 mô hình ML truyền thống

**Mô hình:** Logistic Regression, Decision Tree, Random Forest

---

## 1. Mục tiêu

- Huấn luyện **3 mô hình ML cơ bản** trên dữ liệu đã tiền xử lý từ Notebook 1
- Đánh giá trên tập **validation** và **test**
- Lưu mô hình tốt nhất để so sánh với Deep Learning ở Notebook 4
- Sử dụng metrics: **Accuracy, Precision, Recall, F1-score, AUC-ROC**

Lưu ý: Mất cân bằng lớp đã được xử lý bằng SMOTE trên tập train, validation/test giữ phân bố gốc.

## 2. Import & Load dữ liệu tiền xử lý

In [1]:
import matplotlib
matplotlib.use('Agg')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os

MODEL_DIR = os.path.join('..', 'models')

# Load preprocessed data
data = np.load(os.path.join(MODEL_DIR, 'preprocessed_data.npz'))
X_train_res = data['X_train_res']
y_train_res = data['y_train_res']
X_val = data['X_val']
y_val = data['y_val']
X_test = data['X_test']
y_test = data['y_test']

print('Train (SMOTE):', X_train_res.shape, '| Val:', X_val.shape, '| Test:', X_test.shape)
print('Train 0/1:', sum(y_train_res==0), '/', sum(y_train_res==1))
print('Val 0/1:', sum(y_val==0), '/', sum(y_val==1))
print('Test 0/1:', sum(y_test==0), '/', sum(y_test==1))

Train (SMOTE): (122728, 8) | Val: (14422, 8) | Test: (14422, 8)
Train 0/1: 61364 / 61364
Val 0/1: 13150 / 1272
Test 0/1: 13150 / 1272


In [2]:
# Load feature names
feature_names = joblib.load(os.path.join(MODEL_DIR, 'feature_names.pkl'))
feature_names

['age',
 'hypertension',
 'heart_disease',
 'bmi',
 'HbA1c_level',
 'blood_glucose_level',
 'gender_enc',
 'smoking_enc']

## 3. Đánh giá mô hình — Hàm helper

In [3]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix,
                             classification_report, RocCurveDisplay)
from sklearn.model_selection import cross_val_score

def evaluate_model(model, X_train, y_train, X_val, y_val, X_test, y_test, name):
    """Huấn luyện và đánh giá mô hình trên train/val/test."""
    model.fit(X_train, y_train)
    
    # Predict
    y_pred_val = model.predict(X_val)
    y_pred_test = model.predict(X_test)
    
    # Probabilities (cho AUC)
    y_prob_val = model.predict_proba(X_val)[:, 1] if hasattr(model, 'predict_proba') else None
    y_prob_test = model.predict_proba(X_test)[:, 1] if hasattr(model, 'predict_proba') else None
    
    results = {
        'name': name,
        'val_acc': accuracy_score(y_val, y_pred_val),
        'val_prec': precision_score(y_val, y_pred_val, zero_division=0),
        'val_rec': recall_score(y_val, y_pred_val, zero_division=0),
        'val_f1': f1_score(y_val, y_pred_val, zero_division=0),
        'test_acc': accuracy_score(y_test, y_pred_test),
        'test_prec': precision_score(y_test, y_pred_test, zero_division=0),
        'test_rec': recall_score(y_test, y_pred_test, zero_division=0),
        'test_f1': f1_score(y_test, y_pred_test, zero_division=0),
        'y_pred_test': y_pred_test,
        'y_prob_test': y_prob_test,
        'model': model
    }
    if y_prob_val is not None:
        results['val_auc'] = roc_auc_score(y_val, y_prob_val)
        results['test_auc'] = roc_auc_score(y_test, y_prob_test)
    
    return results

def print_metrics(results, prefix=''):
    print(f"{prefix}{results['name']}")
    print(f"  Validation:  Acc={results['val_acc']:.4f}  Prec={results['val_prec']:.4f}  Rec={results['val_rec']:.4f}  F1={results['val_f1']:.4f}  AUC={results.get('val_auc',0):.4f}")
    print(f"  Test:        Acc={results['test_acc']:.4f}  Prec={results['test_prec']:.4f}  Rec={results['test_rec']:.4f}  F1={results['test_f1']:.4f}  AUC={results.get('test_auc',0):.4f}")
    print()

## 4. Mô hình 1: Logistic Regression (Baseline tuyến tính)

Logistic Regression là baseline tốt cho bài toán phân loại nhị phân. Được sử dụng trong Assignment 02, ta sẽ so sánh lại ở đây.

In [4]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(
    random_state=42,
    max_iter=1000,
    class_weight=None,  # SMOTE đã cân bằng
    solver='lbfgs',
    C=1.0
)

lr_results = evaluate_model(lr, X_train_res, y_train_res, X_val, y_val, X_test, y_test, 'Logistic Regression')
print_metrics(lr_results)

Logistic Regression
  Validation:  Acc=0.8860  Prec=0.4289  Rec=0.8821  F1=0.5772  AUC=0.9623
  Test:        Acc=0.8846  Prec=0.4249  Rec=0.8734  F1=0.5716  AUC=0.9591



### 4.1. Classification report & Confusion matrix (Test)

In [5]:
print('=== TEST Classification Report ===')
print(classification_report(y_test, lr_results['y_pred_test'], target_names=['No Diabetes', 'Diabetes']))

cm = confusion_matrix(y_test, lr_results['y_pred_test'])
fig, ax = plt.subplots(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['No Diabetes', 'Diabetes'],
            yticklabels=['No Diabetes', 'Diabetes'])
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
ax.set_title('Confusion Matrix - Logistic Regression (Test)')
plt.tight_layout(); plt.show()

=== TEST Classification Report ===
              precision    recall  f1-score   support

 No Diabetes       0.99      0.89      0.93     13150
    Diabetes       0.42      0.87      0.57      1272

    accuracy                           0.88     14422
   macro avg       0.71      0.88      0.75     14422
weighted avg       0.94      0.88      0.90     14422



## 5. Mô hình 2: Decision Tree

Decision Tree có khả năng mô hình hóa phi tuyến tính, dễ giải thích. Sử dụng entropy hoặc gini làm criterion.

In [6]:
from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier(
    random_state=42,
    criterion='entropy',
    max_depth=10,
    min_samples_split=10,
    min_samples_leaf=5,
    class_weight=None
)

dt_results = evaluate_model(dt, X_train_res, y_train_res, X_val, y_val, X_test, y_test, 'Decision Tree')
print_metrics(dt_results)

Decision Tree
  Validation:  Acc=0.8999  Prec=0.4641  Rec=0.8679  F1=0.6048  AUC=0.9706
  Test:        Acc=0.8988  Prec=0.4613  Rec=0.8750  F1=0.6041  AUC=0.9712



### 5.1. Classification report & Confusion matrix (Test)

In [7]:
print('=== TEST Classification Report ===')
print(classification_report(y_test, dt_results['y_pred_test'], target_names=['No Diabetes', 'Diabetes']))

cm = confusion_matrix(y_test, dt_results['y_pred_test'])
fig, ax = plt.subplots(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['No Diabetes', 'Diabetes'],
            yticklabels=['No Diabetes', 'Diabetes'])
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
ax.set_title('Confusion Matrix - Decision Tree (Test)')
plt.tight_layout(); plt.show()

=== TEST Classification Report ===
              precision    recall  f1-score   support

 No Diabetes       0.99      0.90      0.94     13150
    Diabetes       0.46      0.88      0.60      1272

    accuracy                           0.90     14422
   macro avg       0.72      0.89      0.77     14422
weighted avg       0.94      0.90      0.91     14422



### 5.2. Feature Importance

In [8]:
dt_importance = pd.DataFrame({
    'feature': feature_names,
    'importance': dt.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(8, 5))
sns.barplot(data=dt_importance, x='importance', y='feature')
plt.title('Feature Importance - Decision Tree')
plt.tight_layout(); plt.show()

print(dt_importance.to_string(index=False))

            feature  importance
        HbA1c_level    0.532298
blood_glucose_level    0.332852
                age    0.109768
                bmi    0.022329
        smoking_enc    0.001545
       hypertension    0.001207
      heart_disease    0.000000
         gender_enc    0.000000


## 6. Mô hình 3: Random Forest

Random Forest là ensemble của nhiều Decision Tree, thường cho hiệu năng tốt và ổn định hơn single tree.

In [9]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    max_depth=12,
    min_samples_split=10,
    min_samples_leaf=5,
    n_jobs=-1,
    class_weight=None
)

rf_results = evaluate_model(rf, X_train_res, y_train_res, X_val, y_val, X_test, y_test, 'Random Forest')
print_metrics(rf_results)

Random Forest
  Validation:  Acc=0.9200  Prec=0.5279  Rec=0.8774  F1=0.6592  AUC=0.9742
  Test:        Acc=0.9191  Prec=0.5253  Rec=0.8577  F1=0.6515  AUC=0.9740



### 6.1. Classification report & Confusion matrix (Test)

In [10]:
print('=== TEST Classification Report ===')
print(classification_report(y_test, rf_results['y_pred_test'], target_names=['No Diabetes', 'Diabetes']))

cm = confusion_matrix(y_test, rf_results['y_pred_test'])
fig, ax = plt.subplots(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['No Diabetes', 'Diabetes'],
            yticklabels=['No Diabetes', 'Diabetes'])
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
ax.set_title('Confusion Matrix - Random Forest (Test)')
plt.tight_layout(); plt.show()

=== TEST Classification Report ===
              precision    recall  f1-score   support

 No Diabetes       0.99      0.93      0.95     13150
    Diabetes       0.53      0.86      0.65      1272

    accuracy                           0.92     14422
   macro avg       0.76      0.89      0.80     14422
weighted avg       0.94      0.92      0.93     14422



### 6.2. Feature Importance

In [11]:
rf_importance = pd.DataFrame({
    'feature': feature_names,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(8, 5))
sns.barplot(data=rf_importance, x='importance', y='feature')
plt.title('Feature Importance - Random Forest')
plt.tight_layout(); plt.show()

print(rf_importance.to_string(index=False))

            feature  importance
        HbA1c_level    0.397153
blood_glucose_level    0.295924
                age    0.168593
                bmi    0.068025
       hypertension    0.030389
        smoking_enc    0.026809
      heart_disease    0.010832
         gender_enc    0.002276


## 7. So sánh 3 mô hình ML cơ bản

In [12]:
# Thu thập kết quả
all_results = [lr_results, dt_results, rf_results]

# Bảng so sánh
comp = pd.DataFrame([{
    'Model': r['name'],
    'Val_Acc': r['val_acc'],
    'Val_Prec': r['val_prec'],
    'Val_Rec': r['val_rec'],
    'Val_F1': r['val_f1'],
    'Val_AUC': r.get('val_auc', 0),
    'Test_Acc': r['test_acc'],
    'Test_Prec': r['test_prec'],
    'Test_Rec': r['test_rec'],
    'Test_F1': r['test_f1'],
    'Test_AUC': r.get('test_auc', 0),
} for r in all_results])

pd.set_option('display.float_format', '{:.4f}'.format)
print(comp.to_string(index=False))

              Model  Val_Acc  Val_Prec  Val_Rec  Val_F1  Val_AUC  Test_Acc  Test_Prec  Test_Rec  Test_F1  Test_AUC
Logistic Regression   0.8860    0.4289   0.8821  0.5772   0.9623    0.8846     0.4249    0.8734   0.5716    0.9591
      Decision Tree   0.8999    0.4641   0.8679  0.6048   0.9706    0.8988     0.4613    0.8750   0.6041    0.9712
      Random Forest   0.9200    0.5279   0.8774  0.6592   0.9742    0.9191     0.5253    0.8577   0.6515    0.9740


### 7.1. Biểu đồ so sánh F1 & AUC (Validation)

In [13]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

x = range(len(comp))
width = 0.35

axes[0].bar([i - width/2 for i in x], comp['Val_F1'], width, label='Val F1', color='#2ca02c')
axes[0].bar([i + width/2 for i in x], comp['Test_F1'], width, label='Test F1', color='#98df8a')
axes[0].set_xticks(x)
axes[0].set_xticklabels(comp['Model'], rotation=15)
axes[0].set_ylabel('F1-score')
axes[0].set_title('F1-score: Validation vs Test')
axes[0].legend()
axes[0].set_ylim(0, 1)

axes[1].bar([i - width/2 for i in x], comp['Val_AUC'], width, label='Val AUC', color='#1f77b4')
axes[1].bar([i + width/2 for i in x], comp['Test_AUC'], width, label='Test AUC', color='#aec7e8')
axes[1].set_xticks(x)
axes[1].set_xticklabels(comp['Model'], rotation=15)
axes[1].set_ylabel('AUC-ROC')
axes[1].set_title('AUC-ROC: Validation vs Test')
axes[1].legend()
axes[1].set_ylim(0, 1)

plt.tight_layout(); plt.show()

### 7.2. ROC Curves trên Test set

In [14]:
fig, ax = plt.subplots(figsize=(8, 6))
for r in all_results:
    if r['y_prob_test'] is not None:
        RocCurveDisplay.from_predictions(y_test, r['y_prob_test'], name=r['name'], ax=ax)
ax.plot([0, 1], [0, 1], 'k--', label='Random (AUC=0.5)')
ax.set_title('ROC Curves - 3 ML Models (Test)')
plt.tight_layout(); plt.show()

## 8. Lưu mô hình tốt nhất cho so sánh sau

Lưu cả 3 mô hình để tái sử dụng trong Notebook 4 (So sánh ML vs DL).

In [15]:
# Chọn mô hình tốt nhất theo Test F1
best = max(all_results, key=lambda x: x['test_f1'])
print(f"Best model by Test F1: {best['name']} (F1={best['test_f1']:.4f})")

# Lưu tất cả
for r in all_results:
    joblib.dump(r['model'], os.path.join(MODEL_DIR, f"diabetes_{r['name'].lower().replace(' ', '_')}.pkl"))

# Lưu kết quả so sánh
comp.to_csv(os.path.join(MODEL_DIR, 'diabetes_ml_comparison.csv'), index=False)
print('✅ Đã lưu 3 models + bảng so sánh vào', MODEL_DIR)

Best model by Test F1: Random Forest (F1=0.6515)
✅ Đã lưu 3 models + bảng so sánh vào ../models


### Nhận xét các mô hình ML
- Với dữ liệu tabular và mất cân bằng lớp, các mô hình cây/ensemble thường có khả năng bắt quan hệ phi tuyến và tương tác giữa các yếu tố sức khỏe tốt hơn mô hình tuyến tính.
- Khi lựa chọn mô hình, cần ưu tiên F1 hoặc Recall của lớp bệnh nếu mục tiêu là giảm bỏ sót ca dương tính; Accuracy cao nhưng Recall thấp có thể bị ảnh hưởng bởi lớp không bệnh chiếm đa số.

## 9. Kết luận (3 ML Models)

- **Logistic Regression**: Baseline tuyến tính, nhanh, interpretable.
- **Decision Tree**: Phi tuyến, dễ hiểu nhưng dễ overfit nếu không điều chỉnh depth.
- **Random Forest**: Ensemble mạnh, ít overfit hơn, thường cho F1/AUC cao nhất.

Mô hình ML tốt nhất (theo Test F1) sẽ được so sánh với Deep Learning ở Notebook 4.

**Tiếp theo: Notebook 3 — Deep Learning (NumPy từ đầu + MLP PyTorch + 3 thí nghiệm bắt buộc)**